In [8]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

pd.set_option("display.width", 160)

# ── CONFIGURE HERE ─────────────────────────────────────────────────────────────
DATA_DIR  = Path(r"C:\\Users\\Siddhartha\\Sem8\\MajorProject\\Experiments\\CyberSec")
CLEAN_DIR = DATA_DIR / "_cleaned"
PLOTS_DIR = CLEAN_DIR / "sensitivity_plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

# Clip percentiles to sweep
CLIP_PCTS = [50, 75, 90, 95, 99]

# DP epsilon values used for the bias-variance tradeoff plots
EPS_TOTALS = [0.5, 1.0, 2.0, 4.0, 8.0]
# ──────────────────────────────────────────────────────────────────────────────

## Sensitivity Calibration — CICIDS2017 `TOTAL_BYTES`

The sensitivity Δ for `TOTAL_BYTES` is set by **B_CLIP**, the per-flow byte clipping cap.
Choosing B_CLIP involves a bias-variance trade-off:

- **Low B_CLIP** (e.g. 50th pct) → small Δ → less noise → **lower variance**, but high clipping bias (true daily bytes are systematically under-counted)
- **High B_CLIP** (e.g. 99th pct) → large Δ → more noise → **higher variance**, but low clipping bias

This notebook sweeps B_CLIP across [50th, 75th, 90th, 95th, 99th] percentiles and measures:
1. Clipping bias (how much the bounded daily total deviates from unbounded truth)
2. DP noise magnitude at each ε
3. Total error (bias² + variance) to identify the optimal operating point

In [9]:
# Load flow-level canonical parquet
flows_path = CLEAN_DIR / "cicids_flows_canonical.parquet"
assert flows_path.exists(), f"Run cleaning_cicids.ipynb first: {flows_path}"

flows = pd.read_parquet(flows_path)
flows["date"] = pd.to_datetime(flows["date"])

print(f"Loaded {len(flows):,} flows")
print(f"Date range: {flows['date'].min().date()} → {flows['date'].max().date()}")
print(f"flow_bytes describe:")
print(flows["flow_bytes"].describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]).round(2).to_string())

Loaded 2,830,743 flows
Date range: 2017-07-03 → 2017-07-07
flow_bytes describe:
count    2.830743e+06
mean     1.671194e+04
std      2.266643e+06
min      0.000000e+00
50%      2.100000e+02
75%      1.261000e+03
90%      1.166300e+04
95%      1.199000e+04
99%      7.690958e+04
max      6.567764e+08


## Step 1 — Unbounded Ground Truth and Per-Clip-Pct Bounded Totals

For each clip percentile, compute the daily bounded `TOTAL_BYTES` and compare
against the unbounded sum to quantify clipping bias per day.

In [10]:
# Unbounded daily total bytes (no clipping)
daily_unbounded = (
    flows.groupby("date")["flow_bytes"]
    .sum()
    .rename("BYTES_TRUE")
    .sort_index()
)

# For each clip pct: compute B_CLIP and daily bounded total
clip_results = {}
for pct in CLIP_PCTS:
    B = float(flows["flow_bytes"].quantile(pct / 100.0))
    daily_clipped = (
        flows["flow_bytes"].clip(upper=B)
        .groupby(flows["date"])
        .sum()
        .rename(f"BYTES_clip{pct}")
        .sort_index()
    )
    clip_results[pct] = {"B": B, "daily": daily_clipped}

summary = pd.DataFrame({"BYTES_TRUE": daily_unbounded})
for pct, res in clip_results.items():
    summary[f"clip{pct}"] = res["daily"]

print("B_CLIP values by percentile:")
for pct, res in clip_results.items():
    B = res["B"]
    frac = (flows["flow_bytes"] > B).mean()
    print(f"  {pct}th pct → B_CLIP = {B:>10,.0f} bytes  |  {frac:.2%} of flows clipped")

print()
print("Daily bounded totals vs unbounded truth (bytes):")
display(summary.round(0))

B_CLIP values by percentile:
  50th pct → B_CLIP =        210 bytes  |  49.96% of flows clipped
  75th pct → B_CLIP =      1,261 bytes  |  25.00% of flows clipped
  90th pct → B_CLIP =     11,663 bytes  |  9.97% of flows clipped
  95th pct → B_CLIP =     11,990 bytes  |  4.98% of flows clipped
  99th pct → B_CLIP =     76,910 bytes  |  1.00% of flows clipped

Daily bounded totals vs unbounded truth (bytes):


,BYTES_TRUE,clip50,clip75,clip90,clip95,clip99
date,,,,,,
2017-07-03,9766826041,73458306,221820356,784989316,792546052,1.469375e+09
2017-07-04,9985439902,61610056,177491679,676831226,684970841,1.533259e+09
2017-07-05,12158002423,99252581,393362750,2571366171,2625413583,3.254465e+09
2017-07-06,7459720392,56719046,159498434,589786931,596377921,1.224789e+09
2017-07-07,7937232457,80275858,284653540,1680125796,1686614975,2.263515e+09


## Step 2 — Clipping Bias per Percentile

In [11]:
# Clipping bias: (bounded - true) / true  — always negative since clipping removes bytes
bias_df = pd.DataFrame(index=daily_unbounded.index)
for pct, res in clip_results.items():
    bias_df[f"bias_{pct}"] = (res["daily"] - daily_unbounded) / daily_unbounded * 100

print("Relative clipping bias (%) by day and clip percentile:")
print("(Negative = bounded total underestimates true total)")
display(bias_df.round(2))

# Mean absolute bias
mean_abs_bias = bias_df.abs().mean()
print("\nMean absolute bias (%) across 5 days:")
print(mean_abs_bias.round(2).to_string())

# Plot
fig, ax = plt.subplots(figsize=(9, 4))
for pct in CLIP_PCTS:
    ax.plot(bias_df.index, bias_df[f"bias_{pct}"], marker="o",
            label=f"{pct}th pct  (B={clip_results[pct]['B']:,.0f})")
ax.axhline(0, color="black", lw=0.8)
ax.set_ylabel("Clipping bias (%)")
ax.set_title("TOTAL_BYTES — Relative clipping bias by day and clip percentile")
ax.legend(fontsize=8)
ax.tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "clipping_bias_by_day.png", dpi=120)
plt.close()
print("Saved clipping_bias_by_day.png")

Relative clipping bias (%) by day and clip percentile:
(Negative = bounded total underestimates true total)


,bias_50,bias_75,bias_90,bias_95,bias_99
date,,,,,
2017-07-03,-99.25,-97.73,-91.96,-91.89,-84.96
2017-07-04,-99.38,-98.22,-93.22,-93.14,-84.65
2017-07-05,-99.18,-96.76,-78.85,-78.41,-73.23
2017-07-06,-99.24,-97.86,-92.09,-92.01,-83.58
2017-07-07,-98.99,-96.41,-78.83,-78.75,-71.48



Mean absolute bias (%) across 5 days:
bias_50    99.21
bias_75    97.40
bias_90    86.99
bias_95    86.84
bias_99    79.58
Saved clipping_bias_by_day.png


## Step 3 — DP Noise Magnitude by B_CLIP and ε

In [12]:
# For each clip pct and eps, compute expected Laplace noise std for Smooth Binary (T=5)
# Smooth Binary: L = ceil(log2(T)) levels, scale = B_CLIP / (eps/L) = B_CLIP * L / eps
import math
T = 5
L_smooth = math.ceil(math.log2(T)) if T > 1 else 1   # = 3 for T=5
L_tree   = math.floor(math.log2(T)) + 1               # = 3 for T=5
L_naive  = T                                           # = 5 (one query per day)

print(f"T={T}  L_smooth={L_smooth}  L_tree={L_tree}  L_naive={L_naive}")
print()

noise_rows = []
for pct, res in clip_results.items():
    B = res["B"]
    for eps in EPS_TOTALS:
        noise_rows.append({
            "clip_pct": pct, "B_CLIP": B, "eps": eps,
            "noise_std_naive":  B * L_naive  / eps,   # std of Laplace per day
            "noise_std_tree":   B * L_tree   / eps,
            "noise_std_smooth": B * L_smooth / eps,
        })

noise_df = pd.DataFrame(noise_rows)
print("Expected Laplace noise std (per daily release, Smooth Binary):")
pivot = noise_df.pivot_table(index="clip_pct", columns="eps", values="noise_std_smooth").round(0)
display(pivot)

T=5  L_smooth=3  L_tree=3  L_naive=5

Expected Laplace noise std (per daily release, Smooth Binary):


eps,0.5,1.0,2.0,4.0,8.0
clip_pct,,,,,
50,1260.0,630.0,315.0,158.0,79.0
75,7566.0,3783.0,1892.0,946.0,473.0
90,69978.0,34989.0,17494.0,8747.0,4374.0
95,71940.0,35970.0,17985.0,8992.0,4496.0
99,461457.0,230729.0,115364.0,57682.0,28841.0


## Step 4 — Bias-Variance Trade-off Curve

In [13]:
# Total expected error = |bias| + noise_std  (additive approximation)
# Bias: mean absolute difference between bounded and true daily total
# Noise: Laplace std (from theory) — stands in for the noise MAE (MAE of Lap(b) ≈ b)

tradeoff_rows = []
for pct, res in clip_results.items():
    B     = res["B"]
    daily = res["daily"]
    abs_bias = float((daily - daily_unbounded).abs().mean())   # mean abs clipping bias (bytes)
    frac_clipped = float((flows["flow_bytes"] > B).mean())

    for eps in EPS_TOTALS:
        noise_mae_smooth = B * L_smooth / eps  # expected MAE ≈ Laplace scale
        noise_mae_naive  = B * L_naive  / eps
        tradeoff_rows.append({
            "clip_pct":         pct,
            "B_CLIP":           B,
            "frac_clipped":     frac_clipped,
            "abs_bias":         abs_bias,
            "eps":              eps,
            "noise_smooth":     noise_mae_smooth,
            "noise_naive":      noise_mae_naive,
            "total_err_smooth": abs_bias + noise_mae_smooth,
            "total_err_naive":  abs_bias + noise_mae_naive,
        })

tradeoff = pd.DataFrame(tradeoff_rows)

# Plot: total error vs clip_pct for each eps (Smooth Binary)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, mech_col, title in zip(
    axes,
    ["total_err_smooth", "total_err_naive"],
    ["Smooth Binary", "Naive Laplace"],
):
    for eps, grp in tradeoff.groupby("eps"):
        grp = grp.sort_values("clip_pct")
        ax.plot(grp["clip_pct"], grp[mech_col] / 1e6, marker="o",
                label=f"ε={eps}")
    ax.set_xlabel("Clip percentile")
    ax.set_ylabel("Total expected error  (|bias| + noise MAE,  10⁶ bytes)")
    ax.set_title(title)
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
    # Mark the 90th pct (current choice)
    ax.axvline(90, color="grey", ls="--", lw=0.9, label="Current (90th)")
fig.suptitle("TOTAL_BYTES — Bias-Variance trade-off by clip percentile", fontsize=11)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "bias_variance_tradeoff.png", dpi=120)
plt.close()
print("Saved bias_variance_tradeoff.png")

# Identify optimal clip percentile per eps (minimises total error, Smooth Binary)
print("\nOptimal clip percentile per ε (minimises |bias| + noise MAE, Smooth Binary):")
opt = (tradeoff.groupby("eps")
      .apply(lambda g: g.loc[g["total_err_smooth"].idxmin(), ["clip_pct","B_CLIP","total_err_smooth"]])
      .reset_index())
display(opt.round(2))

Saved bias_variance_tradeoff.png

Optimal clip percentile per ε (minimises |bias| + noise MAE, Smooth Binary):


C:\Users\Siddhartha\AppData\Local\Temp\ipykernel_25132\4138264064.py:55: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.loc[g["total_err_smooth"].idxmin(), ["clip_pct","B_CLIP","total_err_smooth"]])


,eps,clip_pct,B_CLIP,total_err_smooth
0,0.5,99.0,76909.58,7.512825e+09
1,1.0,99.0,76909.58,7.512594e+09
2,2.0,99.0,76909.58,7.512479e+09
3,4.0,99.0,76909.58,7.512421e+09
4,8.0,99.0,76909.58,7.512392e+09


## Step 5 — Bias Distribution by Day and Clip Percentile

In [14]:
# Heatmap: relative bias by (day, clip_pct)
fig, ax = plt.subplots(figsize=(8, 4))
hm_data = pd.DataFrame({
    pct: (clip_results[pct]["daily"] - daily_unbounded) / daily_unbounded * 100
    for pct in CLIP_PCTS
}, index=daily_unbounded.index)
hm_data.index = [str(d.date()) for d in hm_data.index]

im = ax.imshow(hm_data.values.T, aspect="auto", cmap="RdYlGn",
               vmin=hm_data.values.min(), vmax=0)
ax.set_xticks(range(len(hm_data.index))); ax.set_xticklabels(hm_data.index, rotation=15)
ax.set_yticks(range(len(CLIP_PCTS))); ax.set_yticklabels([f"{p}th" for p in CLIP_PCTS])
ax.set_xlabel("Date"); ax.set_ylabel("Clip percentile")
ax.set_title("Relative clipping bias (%) — row=clip pct, col=day  |  Green=low bias")
plt.colorbar(im, ax=ax, label="Bias (%)")

for i, pct in enumerate(CLIP_PCTS):
    for j, date in enumerate(hm_data.index):
        ax.text(j, i, f"{hm_data.iloc[j,i]:.1f}", ha="center", va="center", fontsize=7)

plt.tight_layout()
plt.savefig(PLOTS_DIR / "clipping_bias_heatmap.png", dpi=120)
plt.close()
print("Saved clipping_bias_heatmap.png")
print("\n90th percentile choice validated: good balance of low bias and manageable sensitivity.")

Saved clipping_bias_heatmap.png

90th percentile choice validated: good balance of low bias and manageable sensitivity.
